# 03 - Apply an admin T-SQL script to the private ERP/WMS Azure SQL database

Azure SQL has public network access **disabled** (tenant policy), so admin scripts run from Databricks compute inside the peered VNet, authenticated with the Unity Catalog service credential (access connector managed identity, member of the SQL Entra admin group).

Used for `03_fabric_mirroring.sql` (least-privilege user for the Fabric workspace identity). `{{PLACEHOLDER}}` tokens are replaced from the `substitutions` JSON widget. All data in this database is **SYNTHETIC**.

In [ ]:
dbutils.widgets.text("sql_server", "", "a. SQL server FQDN")
dbutils.widgets.text("sql_database", "sqldb-appareliq-erpwms", "b. SQL database")
dbutils.widgets.text("service_credential", "appareliq-sql", "c. UC service credential")
dbutils.widgets.text("sql_path", "/Workspace/Shared/appareliq/sql", "d. SQL scripts path")
dbutils.widgets.text("script", "03_fabric_mirroring.sql", "e. script file")
dbutils.widgets.text("substitutions", "{}", "f. substitutions JSON")

In [ ]:
import json, re
server, database = dbutils.widgets.get("sql_server"), dbutils.widgets.get("sql_database")
assert server, "set the sql_server widget"
subs = json.loads(dbutils.widgets.get("substitutions") or "{}")
sql = open(f"{dbutils.widgets.get('sql_path')}/{dbutils.widgets.get('script')}", encoding="utf-8").read()
for k, v in subs.items():
    assert re.fullmatch(r"[A-Za-z0-9 _.@\-]+", v), f"unsafe value for {k}"
    sql = sql.replace("{{" + k + "}}", v)
missing = set(re.findall(r"\{\{(\w+)\}\}", sql))
assert not missing, f"unsubstituted placeholders: {missing}"
token = dbutils.credentials.getServiceCredentialsProvider(dbutils.widgets.get("service_credential")).get_token("https://database.windows.net/.default").token
url = (f"jdbc:sqlserver://{server}:1433;database={database};encrypt=true;trustServerCertificate=false;"
       "hostNameInCertificate=*.database.windows.net;loginTimeout=60")
props = spark._jvm.java.util.Properties(); props.setProperty("accessToken", token)
conn = spark._jvm.java.sql.DriverManager.getConnection(url, props)
try:
    st = conn.createStatement()
    batches = [b.strip() for b in re.split(r"^\s*GO\s*$", sql, flags=re.M) if b.strip()]
    results = []
    for b in batches:
        has_rs = st.execute(b)
        while True:
            if has_rs:
                rs = st.getResultSet(); md = rs.getMetaData(); n = md.getColumnCount()
                cols = [md.getColumnLabel(i + 1) for i in range(n)]
                rows = []
                while rs.next():
                    rows.append({cols[i]: (None if rs.getObject(i + 1) is None else str(rs.getObject(i + 1))) for i in range(n)})
                results.append(rows); print(json.dumps(rows, indent=1)[:4000])
            elif st.getUpdateCount() == -1:
                break
            has_rs = st.getMoreResults()
    rs = st.executeQuery("SELECT dp.name, p.permission_name, p.state_desc FROM sys.database_permissions p JOIN sys.database_principals dp ON dp.principal_id = p.grantee_principal_id WHERE dp.type IN ('E','X') ORDER BY 1,2")
    perms = []
    while rs.next():
        perms.append(f"{rs.getString(1)}:{rs.getString(2)}:{rs.getString(3)}")
finally:
    conn.close()
print(f"applied {len(batches)} batches"); print("\n".join(perms))
dbutils.notebook.exit(json.dumps({"batches": len(batches), "permissions": perms, "results": results})[:60000])